# Proyecto Monitorización IoT: Flows vs Paquetes

Este notebook compara dos enfoques para la clasificación de tráfico IoT:
1. **Machine Learning (Random Forest)** usando datos de flujos (Flows).
2. **Deep Learning (CNN)** usando datos de paquetes (PCAPs).

El objetivo es comparar la precisión (Accuracy) y el tiempo de entrenamiento de ambos modelos.

In [11]:
# Instalar dependencias necesarias
%pip install pandas scikit-learn matplotlib scapy tensorflow numpy

/bin/bash: line 1: /home/orb/TMA_lab/.venv/bin/python: No such file or directory
Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Imports para ML (Flows)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Imports para DL (Paquetes)
from scapy.all import rdpcap
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout
from tensorflow.keras.utils import to_categorical

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 1. Análisis de Flows (Machine Learning)

En esta sección definimos la función `ejecutar_modelo_flows` que:
1. Carga los archivos CSV de flujos.
2. Realiza un muestreo para limitar la cantidad de datos.
3. Preprocesa los datos (OneHotEncoding, Scaling).
4. Entrena un modelo Random Forest.
5. Devuelve la precisión y el tiempo de entrenamiento.

In [ ]:
import os
import glob
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

DATA_FRAC = 1  # Usar el X% del dataset. Numero entre 0 y 1.
CANTIDAD_META = 1000000  # LA MISMA CIFRA QUE EN PCAPS

def ejecutar_modelo_flows(ruta_datos='/media/orb/SSD 1TB/TMA/UNSW-IoTraffic/data_flows/*.csv'):
    print("\n--- [FLOWS] Iniciando carga y entrenamiento ---")
    
    # 1. Cargar CSVs (Basado en tu script original)
    file_paths = glob.glob(ruta_datos)
    if not file_paths:
        print("Error: No se encontraron CSVs en", ruta_datos)
        return 0, 0

    df_list = []
    for fp in file_paths:
        # Leemos todo como texto al principio o especificamos low_memory=False para evitar el warning
        df = pd.read_csv(fp, low_memory=False)
        # Asumimos que el nombre del archivo es la clase (ej: 'AmazonEcho.csv')
        device_name = os.path.basename(fp).split('.')[0] 
        df['device'] = device_name
        df_list.append(df)


    df_all = pd.concat(df_list, ignore_index=True)

    
    # MUESTREO POR CANTIDAD FIJA
    
    if len(df_all) > CANTIDAD_META:
        print(f"Recortando Flows de {len(df_all)} a {CANTIDAD_META}...")
        df_all = df_all.sample(n=CANTIDAD_META, random_state=42)
    
    """
    # MUESTREO POR PORCENTAJE
    if DATA_FRAC < 1.0:
        print(f"Dataset original: {len(df_all)} flows. Usando el {DATA_FRAC*100}%...")
        # frac=0.2 nos da un 20% aleatorio
        df_all = df_all.sample(frac=DATA_FRAC, random_state=42) 
    """
    print(f"Entrenando con {len(df_all)} flows finales.")
    print(f"Cargados {len(df_all)} flows de {df_all['device'].nunique()} dispositivos.")

    # 2. Limpieza (Usando las columnas que tu script sugería borrar)
    drop_cols = ['time','srcMac','dstMac','ethType','srcIp','dstIp','ipProto','srcPort','dstPort','flowSeqNum']
    # Nota: Si da error porque falta alguna columna, quítala de esta lista
    # Remover las columnas descartadas + la etiqueta para evitar fuga de información
    cols_to_drop = [c for c in drop_cols if c in df_all.columns]
    if 'device' in df_all.columns:
        cols_to_drop.append('device')
    X = df_all.drop(columns=cols_to_drop)
    y = df_all['device']

    # 3. Pipeline (Preprocesamiento + Modelo)
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=['number']).columns.tolist()

    preprocessor = ColumnTransformer(transformers=[
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
        ('scale', StandardScaler(), num_cols)
    ])

    # Usamos parámetros fijos (sin GridSearch) para medir tiempo de un solo entrenamiento
    pipeline = Pipeline([
        ('preproc', preprocessor),
        ('rf', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    # 4. Entrenar y Medir Tiempo
    print("Entrenando Random Forest...")
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    tiempo_total = time.time() - start_time

    # 5. Evaluar
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"--> [FLOWS] Terminado. Tiempo: {tiempo_total:.2f}s | Accuracy: {acc:.4f}")
    return acc, tiempo_total

## 2. Análisis de Paquetes (Deep Learning)

En esta sección definimos las funciones para procesar archivos PCAP y entrenar una red neuronal convolucional (CNN):
1. `pcap_to_matrix`: Convierte paquetes crudos en matrices numéricas.
2. `ejecutar_modelo_pcaps`: Carga los datos, normaliza, entrena la CNN y evalúa.

In [9]:
import os
import glob
import time
import numpy as np
from scapy.all import rdpcap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout
from tensorflow.keras.utils import to_categorical

# --- CONFIGURACIÓN GLOBAL ---
CANTIDAD_META = 1000000    # Objetivo: 100k muestras (Igual que pondremos en Flows)
MAX_LEN = 500             # Subimos de 50 a 500 (50 es muy poco, pierdes info útil)
DATA_FRAC = 1.0           # Nos quedamos todo lo que leamos
# Cálculo: 100.000 / 27 dispositivos = ~3700. Ponemos 4000 para ir sobrados.
MAX_PKTS_PER_FILE = 100000

def pcap_to_matrix(file_path, max_len=MAX_LEN, data_frac=DATA_FRAC, max_pkts_read=MAX_PKTS_PER_FILE):
    """
    Lee un PCAP, convierte los paquetes a matrices numéricas y 
    devuelve solo una muestra aleatoria (data_frac) para ahorrar memoria.
    """
    try:
        # 1. Leer los paquetes (limitado por max_pkts_read para no leer archivos gigantes enteros)
        packets = rdpcap(file_path, count=max_pkts_read)
        
        raw_data = []
        
        # 2. Convertir cada paquete a lista de números
        for pkt in packets:
            # Obtener bytes crudos
            byte_list = list(bytes(pkt))
            
            # Recortar o Rellenar (Padding) para tener tamaño fijo
            if len(byte_list) > max_len:
                byte_list = byte_list[:max_len] # Recortar si sobra
            else:
                byte_list = byte_list + [0] * (max_len - len(byte_list)) # Rellenar con ceros si falta
                
            raw_data.append(byte_list)
            
        # Convertimos a array de Numpy temporalmente
        data_array = np.array(raw_data)
        
        # Si el archivo estaba vacío o no se leyó nada, devolvemos array vacío
        if len(data_array) == 0:
            return np.array([])

        # 3. MUESTREO ALEATORIO (Aquí aplicamos el 20% o lo que definas)
        # Calculamos cuántos paquetes nos quedamos
        n_keep = int(len(data_array) * data_frac)
        
        if n_keep > 0:
            # Elegimos índices aleatorios sin repetir
            indices = np.random.choice(len(data_array), n_keep, replace=False)
            # Devolvemos solo los paquetes seleccionados
            return data_array[indices]
        else:
            # Si el porcentaje es muy bajo y da 0 paquetes, devolvemos vacío
            return np.array([])

    except Exception as e:
        print(f"Error leyendo {file_path}: {e}")
        return np.array([])
def ejecutar_modelo_pcaps(ruta_datos='data_pcaps/*.pcap'):
    print(f"\n--- [PAQUETES/DL] Iniciando carga (Meta: {CANTIDAD_META} muestras) ---")
    
    files = glob.glob(ruta_datos)
    X_list, y_list = [], []
    
    print("Procesando PCAPs...")
    for f in files:
        label = os.path.basename(f).split('.')[0]
        # Usamos las variables globales MAX_LEN, DATA_FRAC, MAX_PKTS_PER_FILE
        data = pcap_to_matrix(f)
        if len(data) > 0:
            X_list.append(data)
            y_list.extend([label] * len(data))
            
    if not X_list: return 0, 0

    X = np.concatenate(X_list)
    # Convertimos y_list a array para poder indexarlo
    y_labels = np.array(y_list) 

    # --- RECORTE EXACTO PARA IGUALDAD DE CONDICIONES ---
    if len(X) > CANTIDAD_META:
        print(f"Recortando de {len(X)} a {CANTIDAD_META} para igualar con Flows...")
        # Elegimos índices aleatorios para mantener la variedad de clases
        indices = np.random.choice(len(X), CANTIDAD_META, replace=False)
        X = X[indices]
        y_labels = y_labels[indices]
    # ---------------------------------------------------

    # Normalizar bytes
    X = X / 255.0 
    
    # Codificar etiquetas
    le = LabelEncoder()
    y_integers = le.fit_transform(y_labels)
    y_onehot = to_categorical(y_integers)
    
    # Reshape para CNN
    X = X.reshape(X.shape[0], X.shape[1], 1)
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.3, random_state=42)

    # Modelo
    model = Sequential([
        Conv1D(32, 3, activation='relu', input_shape=(MAX_LEN, 1)),
        MaxPooling1D(2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(len(le.classes_), activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    print(f"Entrenando CNN con {len(X_train)} muestras...")
    start_time = time.time()
    
    # Epochs: 5 es un buen número para empezar. Si va rápido, sube a 10.
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)
    
    tiempo_total = time.time() - start_time
    
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"--> [PAQUETES] Terminado. Tiempo: {tiempo_total:.2f}s | Accuracy: {acc:.4f}")
    return acc, tiempo_total

## 3. Comparación de Resultados

Ejecutamos ambos modelos y comparamos los resultados gráficamente.
**Nota:** Asegúrate de tener las carpetas `data_flows` y `data_pcaps` con los archivos correspondientes en el mismo directorio que este notebook.

In [ ]:
print("=== PROYECTO MONITORIZACIÓN IOT: FLOWS vs PAQUETES ===")

# 1. Ejecutar Flows
acc_flow, time_flow = ejecutar_modelo_flows()

# 2. Ejecutar Paquetes
acc_pcap, time_pcap = ejecutar_modelo_pcaps()

if time_flow == 0 and time_pcap == 0:
    print("No se generaron datos. Revisa las carpetas data_flows y data_pcaps.")
else:
    # 3. Graficar
    labels = ['Flows (ML)', 'Paquetes (DL)']
    accuracies = [acc_flow, acc_pcap]
    times = [time_flow, time_pcap]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Gráfica Accuracy
    ax1.bar(labels, accuracies, color=['#4CAF50', '#2196F3'])
    ax1.set_title('Comparación de Precisión (Accuracy)')
    ax1.set_ylim(0, 1.1)
    for i, v in enumerate(accuracies):
        ax1.text(i, v + 0.02, f"{v*100:.1f}%", ha='center', fontweight='bold')

    # Gráfica Tiempo
    ax2.bar(labels, times, color=['#4CAF50', '#2196F3'])
    ax2.set_title('Tiempo de Entrenamiento (Segundos)')
    ax2.set_ylabel('Segundos')
    for i, v in enumerate(times):
        ax2.text(i, v, f"{v:.1f}s", ha='center', va='bottom', fontweight='bold')

    plt.suptitle('Análisis de Tráfico IoT: Flows vs Paquetes')
    plt.tight_layout()
    plt.savefig('comparativa_resultados.png')
    plt.show()

    print("\nGráfica guardada como 'comparativa_resultados.png'")